In [ ]:
"""
=============================================================
FUNDUS IMAGE → EXACT COLORFUL NORMALIZED VESSEL IMAGE
Reference: Colorful vessel overlay (blue/red/green channels)
=============================================================
pip install opencv-python numpy matplotlib scipy scikit-image
=============================================================
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage import filters, morphology, exposure
import warnings
warnings.filterwarnings('ignore')


# ─────────────────────────────────────────────
# LOAD IMAGE
# ─────────────────────────────────────────────
def load_image(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# ─────────────────────────────────────────────
# CREATE CIRCULAR MASK (retinal boundary)
# ─────────────────────────────────────────────
def create_circular_mask(h, w):
    cy, cx = h // 2, w // 2
    r = min(cy, cx) - 5
    Y, X = np.ogrid[:h, :w]
    mask = (X - cx)**2 + (Y - cy)**2 <= r**2
    return mask.astype(np.uint8)


# ─────────────────────────────────────────────
# NORMALIZE EACH CHANNEL WITH CLAHE
# ─────────────────────────────────────────────
def clahe_channel(channel, clip=3.0, grid=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=grid)
    return clahe.apply(channel)


# ─────────────────────────────────────────────
# BACKGROUND SUBTRACTION PER CHANNEL
# ─────────────────────────────────────────────
def subtract_background(channel, sigma=30):
    ch = channel.astype(np.float32)
    bg = gaussian_filter(ch, sigma=sigma)
    result = ch - bg + 128.0
    return np.clip(result, 0, 255).astype(np.uint8)


# ─────────────────────────────────────────────
# MAIN: COLORFUL NORMALIZED IMAGE
# ─────────────────────────────────────────────
def create_colorful_normalized(img_rgb):
    """
    Recreates the exact colorful normalized fundus image:
    - Each RGB channel processed independently
    - Background subtracted → CLAHE → recombined
    - Vessels appear dark red/blue, background is teal/green
    """
    H, W = img_rgb.shape[:2]
    circ_mask = create_circular_mask(H, W)

    result = np.zeros_like(img_rgb)

    # Process each channel differently to get the colorful look
    for i, (sigma, clip) in enumerate([(25, 4.0), (20, 3.5), (30, 4.5)]):
        ch = img_rgb[:, :, i]
        # Background subtraction
        bg_sub = subtract_background(ch, sigma=sigma)
        # CLAHE
        enhanced = clahe_channel(bg_sub, clip=clip, grid=(8, 8))
        result[:, :, i] = enhanced

    # Apply circular mask — outside = gray (like the reference image)
    gray_bg = np.full_like(result, 128)
    for c in range(3):
        result[:, :, c] = np.where(circ_mask, result[:, :, c], gray_bg[:, :, c])

    return result


# ─────────────────────────────────────────────
# ENHANCED VERSION: more vivid colors
# ─────────────────────────────────────────────
def boost_colors(normalized_img, saturation=1.8, contrast=1.2):
    """Boost saturation to make vessel colors more vivid."""
    hsv = cv2.cvtColor(normalized_img, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * saturation, 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * contrast, 0, 255)
    boosted = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    return boosted


# ─────────────────────────────────────────────
# VESSEL MAP (dark vessel lines)
# ─────────────────────────────────────────────
def get_vessel_overlay(img_rgb):
    """Extract vessel darkness to darken vessel lines in final image."""
    green = img_rgb[:, :, 1].astype(np.float32)
    bg = gaussian_filter(green, sigma=20)
    vessels = bg - green  # vessels are darker than background
    vessels = np.clip(vessels, 0, None)
    vessels = vessels / (vessels.max() + 1e-8)
    return vessels


# ─────────────────────────────────────────────
# FULL PIPELINE + VISUALIZATION
# ─────────────────────────────────────────────
def run(image_path, save_path="colorful_normalized_fundus.png"):
    print("[→] Loading image...")
    img_rgb = load_image(image_path)
    H, W = img_rgb.shape[:2]

    print("[→] Creating colorful normalized image...")
    normalized = create_colorful_normalized(img_rgb)

    print("[→] Boosting colors...")
    vivid = boost_colors(normalized, saturation=2.0, contrast=1.15)

    # Re-apply circular mask after boost
    circ_mask = create_circular_mask(H, W)
    gray_bg = np.full_like(vivid, 128)
    for c in range(3):
        vivid[:, :, c] = np.where(circ_mask, vivid[:, :, c], gray_bg[:, :, c])

    # ── Side-by-side comparison ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), facecolor='#1a1a1a')

    axes[0].imshow(img_rgb)
    axes[0].set_title("(a) Raw Fundus Image", color='white', fontsize=13, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(vivid)
    axes[1].set_title("(b) Colorful Normalized Image", color='white', fontsize=13, fontweight='bold')
    axes[1].axis('off')

    plt.suptitle("Fundus Normalization — Exact Colorful Output",
                 color='white', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig("comparison.png", dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
    plt.show()

    # ── Save normalized only ──
    cv2.imwrite(save_path, cv2.cvtColor(vivid, cv2.COLOR_RGB2BGR))
    print(f"[✓] Saved: {save_path}")
    print(f"[✓] Saved: comparison.png")

    return vivid


# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    IMAGE_PATH = "fundus.jpg"   # ← আপনার image path

    result = run(IMAGE_PATH)